# HMARL Full Experiment Suite — Colab Pro

This notebook runs the **complete** experiment pipeline on Google Colab Pro:

| # | Experiment | Seeds | Iterations | Est. Time |
|---|-----------|-------|------------|----------|
| 1 | **Baseline** | 1 | 200 | ~8 min |
| 2 | **Multi-seed** | 5 | 100 | ~20 min |
| 3 | **No-sharing ablation** | 3 | 200 | ~24 min |
| 4 | **Weather curriculum** | 3 | 300 | ~36 min |
| 5 | **PBT** (4 workers) | 1 | 100 | ~16 min |
| 6 | **Paper figures** | — | — | ~2 min |
| | **Total** | | | **~1 hr 45 min** |

**Recommended runtime**: `High-RAM` CPU runtime (no GPU needed).

- Runtime → Change runtime type → **High-RAM** (Colab Pro gives you 25–51 GB RAM)
- GPU is not beneficial — the networks are tiny (~100K params) and the bottleneck is CPU-based environment stepping
- Colab Pro gives you better CPUs and longer session limits (up to 24 hrs)

All outputs are saved to Google Drive under `qgi_lab_hmarl_runs/`.

## 1. Setup: Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

GIT_BRANCH = 'main'
REPO_ROOT = Path('/content/qgi_lab_hmarl')
RUNS_ROOT = Path('/content/drive/MyDrive/qgi_lab_hmarl_runs')
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

print('Repo root:', REPO_ROOT)
print('Runs root (Drive):', RUNS_ROOT)

In [ ]:
import shutil
import subprocess
import sys

if REPO_ROOT.exists():
    shutil.rmtree(REPO_ROOT)

subprocess.run(
    ['git', 'clone', 'https://github.com/zhongnz/qgi_lab_hmarl.git', str(REPO_ROOT)],
    check=True,
)
subprocess.run(['git', '-C', str(REPO_ROOT), 'fetch', '--all'], check=True)

checkout = subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', GIT_BRANCH])
if checkout.returncode != 0:
    print(f'Branch {GIT_BRANCH!r} not found; falling back to main.')
    subprocess.run(['git', '-C', str(REPO_ROOT), 'checkout', 'main'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip', '-q'], check=True)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r',
     str(REPO_ROOT / 'requirements.txt'), '-q'],
    check=True,
)
print('\nSetup complete.')

In [ ]:
import os
import platform

import psutil
import torch

print(f'Python:   {sys.version}')
print(f'Platform: {platform.platform()}')
print(f'Torch:    {torch.__version__}')
print(f'CPU cores: {os.cpu_count()}')
print(f'RAM:      {psutil.virtual_memory().total / 1e9:.1f} GB')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')

# CPU is optimal for this workload
DEVICE = 'cpu'

In [ ]:
# Add repo to path and verify imports
sys.path.insert(0, str(REPO_ROOT))

from hmarl_mvp import (
    MaritimeEnv,
    MAPPOConfig,
    MAPPOTrainer,
    PBTConfig,
    PBTTrainer,
    get_default_config,
)
from hmarl_mvp.curriculum import CurriculumScheduler, CurriculumStage
from hmarl_mvp.experiment import run_trained_mappo_trace
from hmarl_mvp.experiment_config import load_experiment_config, run_from_config
from hmarl_mvp.mappo import train_multi_seed
from hmarl_mvp.report import generate_training_report

print('All imports successful.')

## 2. Quick Smoke Test

Verify the environment and training loop work before committing to long runs.

In [ ]:
import time

t0 = time.time()
cfg = get_default_config(num_vessels=2, num_ports=2, docks_per_port=1, rollout_steps=15)
smoke = MAPPOTrainer(
    env_config=cfg,
    mappo_config=MAPPOConfig(rollout_length=4, num_epochs=1, hidden_dims=[16, 16],
                             vessel_hidden_dims=[16, 16]),
    seed=42,
)
r = smoke.collect_rollout()
u = smoke.update()
dt = time.time() - t0

print(f'Smoke test passed in {dt:.1f}s')
print(f'  mean_reward: {r["mean_reward"]:.4f}')
print(f'  vessel policy_loss: {u["vessel"].policy_loss:.4f}')

# Estimate iteration time at full scale
t0 = time.time()
full_cfg = get_default_config()
bench = MAPPOTrainer(env_config=full_cfg, mappo_config=MAPPOConfig(), seed=42)
bench.collect_rollout()
bench.update()
iter_time = time.time() - t0
print(f'\nFull-scale iteration time: {iter_time:.1f}s')
print(f'Estimated time for 200 iterations: {iter_time * 200 / 60:.0f} min')
del smoke, bench

## 3. Shared Configuration & Utilities

In [ ]:
import json

import numpy as np
import pandas as pd

from hmarl_mvp.plotting import (
    plot_multi_seed_curves,
    plot_time_series_diagnostics,
    plot_training_curves,
    plot_training_dashboard,
    plot_reward_decomposition,
    plot_ablation_bar,
)

# Shared base env config
BASE_ENV = {
    'num_vessels': 8,
    'num_ports': 5,
    'docks_per_port': 3,
    'rollout_steps': 69,
    'episode_mode': 'continuous',
    'forecast_source': 'ground_truth',
    'weather_enabled': True,
    'weather_autocorrelation': 0.7,
    'requested_arrival_slack_hours': 1.0,
}

# Shared base MAPPO config
BASE_MAPPO = dict(
    lr=3e-4,
    lr_end=1e-4,
    rollout_length=64,
    num_epochs=4,
    minibatch_size=128,
    clip_eps=0.2,
    gamma=0.99,
    gae_lambda=0.95,
    hidden_dims=[64, 64],
    vessel_hidden_dims=[128, 128],
    entropy_coeff=0.05,
    entropy_coeff_end=0.002,
    max_grad_norm=0.5,
    max_grad_norm_start=5.0,
    grad_norm_warmup_fraction=0.1,
    normalize_observations=True,
    normalize_rewards=True,
    device='cpu',
)


def save_run_artifacts(run_name, trainer, history, out_dir=None, eval_episodes=5):
    """Save all artifacts for a training run to Drive."""
    out = out_dir or (RUNS_ROOT / run_name)
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)

    # History CSV
    df = pd.DataFrame(history)
    df.to_csv(out / 'train_history.csv', index=False)

    # Model checkpoint
    trainer.save_models(str(out / 'final_model'))

    # Evaluation
    eval_result = trainer.evaluate_episodes(num_episodes=eval_episodes)
    with open(out / 'eval_result.json', 'w') as f:
        json.dump(eval_result, f, indent=2, default=str)

    # Trace
    trace_result = run_trained_mappo_trace(trainer, num_steps=int(trainer.cfg['rollout_steps']), return_logs=True)
    if isinstance(trace_result, tuple):
        trace, action_trace, event_log = trace_result
    else:
        trace, action_trace, event_log = trace_result, pd.DataFrame(), pd.DataFrame()
    trace.to_csv(out / 'eval_trace.csv', index=False)
    action_trace.to_csv(out / 'eval_action_trace.csv', index=False)
    event_log.to_csv(out / 'eval_event_log.csv', index=False)

    # Plots
    plot_training_curves(df, out_path=str(out / 'training_curves.png'))
    plot_time_series_diagnostics(trace, out_path=str(out / 'diagnostics.png'), column_group='aggregate')

    print(f'  Saved to {out}')
    return eval_result


print('Utilities ready.')

---
## 4. Experiment 1: Baseline (200 iterations, single seed)

Standard MAPPO training with default hyperparameters.

In [ ]:
print('=== Experiment 1: Baseline ===')
t0 = time.time()

baseline_cfg = MAPPOConfig(**{**BASE_MAPPO, 'total_iterations': 200})
baseline_trainer = MAPPOTrainer(env_config=BASE_ENV, mappo_config=baseline_cfg, seed=42)
baseline_history = baseline_trainer.train(
    num_iterations=200,
    eval_interval=20,
    checkpoint_dir=str(RUNS_ROOT / 'baseline' / 'checkpoints'),
    early_stopping_patience=0,
)

baseline_eval = save_run_artifacts('baseline', baseline_trainer, baseline_history)
baseline_time = time.time() - t0
print(f'Baseline done in {baseline_time / 60:.1f} min')
print(f'  Best reward:  {max(h["mean_reward"] for h in baseline_history):.4f}')
print(f'  Final reward: {baseline_history[-1]["mean_reward"]:.4f}')
pd.DataFrame([baseline_eval['mean']])

---
## 5. Experiment 2: Multi-seed (5 seeds × 100 iterations)

Statistical evaluation across 5 seeds to measure variance.

In [ ]:
print('=== Experiment 2: Multi-seed ===')
t0 = time.time()

SEEDS = [42, 49, 56, 63, 70]
ms_cfg = MAPPOConfig(**{**BASE_MAPPO, 'total_iterations': 100})
ms_out = RUNS_ROOT / 'multi_seed'
ms_out.mkdir(parents=True, exist_ok=True)

ms_result = train_multi_seed(
    env_config=BASE_ENV,
    mappo_config=ms_cfg,
    num_iterations=100,
    seeds=SEEDS,
    eval_interval=10,
    early_stopping_patience=30,
    checkpoint_dir=str(ms_out / 'checkpoints'),
)

# Save per-seed histories
for seed, hist in zip(SEEDS, ms_result['histories']):
    seed_dir = ms_out / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(hist).to_csv(seed_dir / 'metrics.csv', index=False)

# Save aggregate summary
with open(ms_out / 'summary.json', 'w') as f:
    json.dump(ms_result.get('aggregate_summary', {}), f, indent=2, default=str)

# Plot multi-seed learning curves
plot_multi_seed_curves(
    ms_result['histories'],
    label='multi_seed',
    out_path=str(ms_out / 'multi_seed_curves.png'),
)

ms_time = time.time() - t0
print(f'Multi-seed done in {ms_time / 60:.1f} min')

agg = ms_result.get('aggregate_summary', {})
print(f'  Mean best reward: {agg.get("mean_best_reward", "N/A")}')
print(f'  Std best reward:  {agg.get("std_best_reward", "N/A")}')

pd.DataFrame(ms_result['summaries'])

---
## 6. Experiment 3: No-sharing Ablation (3 seeds × 200 iterations)

Per-agent networks (no parameter sharing) — quantifies the benefit of CTDE sharing.

In [ ]:
print('=== Experiment 3: No-sharing Ablation ===')
t0 = time.time()

NS_SEEDS = [42, 49, 56]
ns_cfg = MAPPOConfig(**{**BASE_MAPPO, 'total_iterations': 200, 'parameter_sharing': False})
ns_out = RUNS_ROOT / 'no_sharing'
ns_out.mkdir(parents=True, exist_ok=True)

ns_result = train_multi_seed(
    env_config=BASE_ENV,
    mappo_config=ns_cfg,
    num_iterations=200,
    seeds=NS_SEEDS,
    eval_interval=20,
    early_stopping_patience=0,
    checkpoint_dir=str(ns_out / 'checkpoints'),
)

for seed, hist in zip(NS_SEEDS, ns_result['histories']):
    seed_dir = ns_out / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(hist).to_csv(seed_dir / 'metrics.csv', index=False)

with open(ns_out / 'summary.json', 'w') as f:
    json.dump(ns_result.get('aggregate_summary', {}), f, indent=2, default=str)

ns_time = time.time() - t0
print(f'No-sharing done in {ns_time / 60:.1f} min')
pd.DataFrame(ns_result['summaries'])

---
## 7. Experiment 4: Weather Curriculum (3 seeds × 300 iterations)

Progressive weather severity ramp over 4 stages.

In [ ]:
print('=== Experiment 4: Weather Curriculum ===')
t0 = time.time()

WC_SEEDS = [42, 49, 56]
wc_cfg = MAPPOConfig(**{**BASE_MAPPO, 'lr_end': 5e-5, 'total_iterations': 300})
wc_out = RUNS_ROOT / 'weather_curriculum'

# Curriculum stages: calm → mild → moderate → full
curriculum_stages = [
    CurriculumStage(fraction=0.0,  config_overrides={'weather_enabled': False}),
    CurriculumStage(fraction=0.25, config_overrides={'weather_enabled': True, 'weather_penalty_factor': 0.05}),
    CurriculumStage(fraction=0.50, config_overrides={'weather_enabled': True, 'weather_penalty_factor': 0.10}),
    CurriculumStage(fraction=0.75, config_overrides={'weather_enabled': True, 'weather_penalty_factor': 0.15}),
]

wc_histories = []
wc_summaries = []

for seed in WC_SEEDS:
    print(f'  Seed {seed}...')
    curriculum = CurriculumScheduler(
        target_config=dict(BASE_ENV),
        stages=curriculum_stages,
    )
    trainer = MAPPOTrainer(env_config=BASE_ENV, mappo_config=wc_cfg, seed=seed)
    hist = trainer.train(
        num_iterations=300,
        curriculum=curriculum,
        eval_interval=25,
        early_stopping_patience=40,
        checkpoint_dir=str(wc_out / f'seed_{seed}' / 'checkpoints'),
    )
    wc_histories.append(hist)
    wc_summaries.append(MAPPOTrainer.training_summary(hist))

    seed_dir = wc_out / f'seed_{seed}'
    seed_dir.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(hist).to_csv(seed_dir / 'metrics.csv', index=False)
    trainer.save_models(str(seed_dir / 'final_model'))

# Aggregate
with open(wc_out / 'summary.json', 'w') as f:
    json.dump({'summaries': wc_summaries}, f, indent=2, default=str)

plot_multi_seed_curves(
    wc_histories,
    label='weather_curriculum',
    out_path=str(wc_out / 'weather_curriculum_curves.png'),
)

wc_time = time.time() - t0
print(f'Weather curriculum done in {wc_time / 60:.1f} min')
pd.DataFrame(wc_summaries)

---
## 8. Experiment 5: PBT (4 workers × 100 iterations)

Population-Based Training with sequential workers. PBT tunes lr, entropy, and clip_eps online — rescuing failing seeds and adapting hyperparameters mid-training.

In [ ]:
print('=== Experiment 5: PBT ===')
t0 = time.time()

pbt_out = RUNS_ROOT / 'pbt'
pbt_out.mkdir(parents=True, exist_ok=True)

pbt_mappo_cfg = MAPPOConfig(**{**BASE_MAPPO})
pbt_cfg = PBTConfig(
    population_size=4,
    interval=10,
    fraction_top=0.25,
    fraction_bottom=0.25,
    perturb_factor=1.2,
    lr_min=1e-5,
    lr_max=1e-3,
    entropy_min=0.001,
    entropy_max=0.1,
    clip_eps_min=0.1,
    clip_eps_max=0.3,
)

pbt_trainer = PBTTrainer(
    env_config=BASE_ENV,
    mappo_config=pbt_mappo_cfg,
    pbt_config=pbt_cfg,
    base_seed=42,
)

pbt_result = pbt_trainer.train(
    total_iterations=100,
    checkpoint_dir=str(pbt_out / 'checkpoints'),
)

# Save PBT results
with open(pbt_out / 'pbt_result.json', 'w') as f:
    json.dump({
        'best_worker_idx': pbt_result['best_worker_idx'],
        'best_mean_reward': pbt_result['best_mean_reward'],
        'total_rounds': pbt_result['total_rounds'],
        'total_time': pbt_result['total_time'],
        'final_hyperparams': pbt_result['final_hyperparams'],
        'exploit_log': pbt_result['exploit_log'],
    }, f, indent=2, default=str)

# Save per-worker reward curves
pbt_df_rows = []
for w_idx, rewards in enumerate(pbt_result['per_worker_rewards']):
    for it, r in enumerate(rewards):
        pbt_df_rows.append({'worker': w_idx, 'iteration': it, 'mean_reward': r})
pbt_df = pd.DataFrame(pbt_df_rows)
pbt_df.to_csv(pbt_out / 'pbt_rewards.csv', index=False)

# Save best worker model
best_worker = pbt_trainer.best_worker
best_worker.save_models(str(pbt_out / 'best_model'))

# Evaluate best worker
pbt_eval = best_worker.evaluate_episodes(num_episodes=5)
with open(pbt_out / 'eval_result.json', 'w') as f:
    json.dump(pbt_eval, f, indent=2, default=str)

pbt_time = time.time() - t0
print(f'PBT done in {pbt_time / 60:.1f} min')
print(f'  Best worker: {pbt_result["best_worker_idx"]}')
print(f'  Best mean reward: {pbt_result["best_mean_reward"]:.4f}')
print(f'  Final hyperparams:')
for i, hp in enumerate(pbt_result['final_hyperparams']):
    print(f'    Worker {i}: lr={hp["lr"]:.6f}  ent={hp["entropy_coeff"]:.4f}  eps={hp["clip_eps"]:.3f}')

In [ ]:
# Plot PBT worker reward curves
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reward curves per worker
ax = axes[0]
for w_idx in range(pbt_cfg.population_size):
    sub = pbt_df[pbt_df['worker'] == w_idx]
    # Smooth with rolling mean
    smoothed = sub['mean_reward'].rolling(window=5, min_periods=1).mean()
    ax.plot(sub['iteration'], smoothed, label=f'Worker {w_idx}', alpha=0.8)
ax.set_xlabel('Iteration')
ax.set_ylabel('Mean Reward (smoothed)')
ax.set_title('PBT Worker Reward Curves')
ax.legend()
ax.grid(True, alpha=0.3)

# Hyperparameter evolution
ax = axes[1]
for w_idx in range(pbt_cfg.population_size):
    hp_hist = pbt_result['hyperparam_history'][w_idx]
    lrs = [h['lr'] for h in hp_hist]
    ax.plot(range(len(lrs)), lrs, label=f'Worker {w_idx}', alpha=0.8)
ax.set_xlabel('Iteration')
ax.set_ylabel('Learning Rate')
ax.set_title('PBT Learning Rate Evolution')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(pbt_out / 'pbt_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('PBT analysis saved.')

---
## 9. Generate Paper Figures

Run the figure generation script using all experiment outputs.

In [ ]:
print('=== Generating Paper Figures ===')
t0 = time.time()

fig_out = RUNS_ROOT / 'figures'
fig_out.mkdir(parents=True, exist_ok=True)

# Figure 1: Baseline training curves
baseline_df = pd.DataFrame(baseline_history)
plot_training_curves(baseline_df, out_path=str(fig_out / 'fig1_training_curves.png'))
print('  fig1: training curves')

# Figure 2: Multi-seed learning curves
plot_multi_seed_curves(
    ms_result['histories'],
    label='MAPPO (5 seeds)',
    out_path=str(fig_out / 'fig2_multi_seed_curves.png'),
)
print('  fig2: multi-seed curves')

# Figure 3: Training dashboard (baseline)
plot_training_dashboard(baseline_df, out_path=str(fig_out / 'fig3_training_dashboard.png'))
print('  fig3: training dashboard')

# Figure 4: Reward decomposition
plot_reward_decomposition(baseline_df, out_path=str(fig_out / 'fig4_reward_decomposition.png'))
print('  fig4: reward decomposition')

# Figure 5: No-sharing ablation comparison
ablation_data = {}
ms_rewards = [s.get('best_mean_reward', 0) for s in ms_result['summaries']]
ns_rewards = [s.get('best_mean_reward', 0) for s in ns_result['summaries']]
ablation_data['Shared'] = ms_rewards
ablation_data['Per-agent'] = ns_rewards
plot_ablation_bar(ablation_data, out_path=str(fig_out / 'fig5_sharing_ablation.png'))
print('  fig5: sharing ablation')

# Figure 7: Weather curriculum curves
plot_multi_seed_curves(
    wc_histories,
    label='Weather Curriculum',
    out_path=str(fig_out / 'fig7_weather_curriculum.png'),
)
print('  fig7: weather curriculum')

# PBT figure
import shutil
shutil.copy(str(pbt_out / 'pbt_analysis.png'), str(fig_out / 'fig_pbt_analysis.png'))
print('  fig_pbt: PBT analysis')

fig_time = time.time() - t0
print(f'\nFigures done in {fig_time:.0f}s')
print(f'Saved to: {fig_out}')
print(f'Files: {sorted(f.name for f in fig_out.glob("*.png"))}')

---
## 10. Summary & Comparison Table

In [ ]:
total_time = baseline_time + ms_time + ns_time + wc_time + pbt_time

# Comparison table
rows = []

# Baseline
bl_summary = MAPPOTrainer.training_summary(baseline_history)
rows.append({
    'Experiment': 'Baseline',
    'Seeds': 1,
    'Iterations': 200,
    'Best Reward': f"{bl_summary.get('best_mean_reward', 0):.4f}",
    'Final Reward': f"{bl_summary.get('final_mean_reward', 0):.4f}",
    'Time (min)': f"{baseline_time / 60:.1f}",
})

# Multi-seed
ms_best = [s.get('best_mean_reward', 0) for s in ms_result['summaries']]
rows.append({
    'Experiment': 'Multi-seed (shared)',
    'Seeds': 5,
    'Iterations': 100,
    'Best Reward': f"{np.mean(ms_best):.4f} ± {np.std(ms_best):.4f}",
    'Final Reward': '-',
    'Time (min)': f"{ms_time / 60:.1f}",
})

# No-sharing
ns_best = [s.get('best_mean_reward', 0) for s in ns_result['summaries']]
rows.append({
    'Experiment': 'No-sharing (per-agent)',
    'Seeds': 3,
    'Iterations': 200,
    'Best Reward': f"{np.mean(ns_best):.4f} ± {np.std(ns_best):.4f}",
    'Final Reward': '-',
    'Time (min)': f"{ns_time / 60:.1f}",
})

# Weather curriculum
wc_best = [s.get('best_mean_reward', 0) for s in wc_summaries]
rows.append({
    'Experiment': 'Weather curriculum',
    'Seeds': 3,
    'Iterations': 300,
    'Best Reward': f"{np.mean(wc_best):.4f} ± {np.std(wc_best):.4f}",
    'Final Reward': '-',
    'Time (min)': f"{wc_time / 60:.1f}",
})

# PBT
rows.append({
    'Experiment': 'PBT (4 workers)',
    'Seeds': '4 pop',
    'Iterations': 100,
    'Best Reward': f"{pbt_result['best_mean_reward']:.4f}",
    'Final Reward': '-',
    'Time (min)': f"{pbt_time / 60:.1f}",
})

comparison = pd.DataFrame(rows)
comparison.to_csv(RUNS_ROOT / 'experiment_comparison.csv', index=False)

print(f'\nTotal runtime: {total_time / 60:.1f} min ({total_time / 3600:.1f} hrs)')
print(f'All outputs saved to: {RUNS_ROOT}')
comparison

In [ ]:
# Final Drive sync confirmation
print('Drive contents:')
for p in sorted(RUNS_ROOT.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {p.relative_to(RUNS_ROOT)}  ({size_mb:.1f} MB)')

---
## Runtime Notes

**Colab Pro recommended settings:**
- Runtime type: **High-RAM** (not GPU)
- Colab Pro gives: ~25 GB RAM, better CPUs, 24-hr sessions
- Background execution enabled (Colab Pro+)

**If the session disconnects:**
- All completed experiments are already saved to Drive
- Re-run only the cells for unfinished experiments
- The smoke test cell helps verify the environment is still set up

**Memory usage:** ~1.5 GB per trainer, PBT peak ~6 GB for 4 workers. Well within the 25 GB Colab Pro limit.